# RQ2: Model Comparison
**Research Question:** Which supervised learning model achieves the best predictive performance for Revenue_Generated, and how does it compare with competing methods?

**Models:** Linear Regression, Random Forest, XGBoost, SVR

In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

In [4]:
DATA_PATH = '/kaggle/input/datasets/karansridhar/karan-ue-ml-assignment-dataset/marketing_and_product_performance.csv'
df = pd.read_csv(DATA_PATH)
TARGET = 'Revenue_Generated'

id_cols = [c for c in df.columns if 'ID' in c or 'id' in c.lower()]
df_model = df.drop(columns=id_cols, errors='ignore').dropna(subset=[TARGET])

cat_cols = df_model.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].fillna('Unknown').astype(str))

df_model = df_model.fillna(df_model.median(numeric_only=True))
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)
print('Data ready:', X_train.shape)

Data ready: (8000, 11)


In [5]:
models = {
    'Linear Regression': (LinearRegression(), True),
    'Random Forest':     (RandomForestRegressor(n_estimators=100, random_state=42), False),
    'XGBoost':           (XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0), False),
    'SVR':               (SVR(kernel='rbf', C=10), True)
}

results = []
for name, (model, use_scaled) in models.items():
    Xtr = X_tr_sc if use_scaled else X_train
    Xte = X_te_sc  if use_scaled else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R²': round(r2,4)})
    print(f'{name:22s} | MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')

df_results = pd.DataFrame(results)
df_results.to_csv('tables/RQ2_Table2_Model_Comparison.csv', index=False)
print(df_results)

Linear Regression      | MAE=24745.4005  RMSE=28590.8029  R²=-0.0025
Random Forest          | MAE=25035.6351  RMSE=29029.5867  R²=-0.0335
XGBoost                | MAE=25223.6129  RMSE=29267.4599  R²=-0.0505
SVR                    | MAE=24727.8079  RMSE=28588.1388  R²=-0.0023
               Model         MAE        RMSE      R²
0  Linear Regression  24745.4005  28590.8029 -0.0025
1      Random Forest  25035.6351  29029.5867 -0.0335
2            XGBoost  25223.6129  29267.4599 -0.0505
3                SVR  24727.8079  28588.1388 -0.0023


In [6]:
# Figure 2 — Horizontal bar chart (R² ranking)
df_sorted = df_results.sort_values('R²', ascending=True)
colors = ['#E53935' if v < df_results['R²'].max() else '#43A047' for v in df_sorted['R²']]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, metric, color in zip(axes, ['MAE','RMSE','R²'], ['#EF5350','#FB8C00','#42A5F5']):
    df_plot = df_results.sort_values(metric, ascending=(metric != 'R²'))
    ax.barh(df_plot['Model'], df_plot[metric], color=color, edgecolor='white')
    ax.set_xlabel(metric, fontsize=11)
    ax.set_title(f'Ranked by {metric}', fontsize=11)
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('Figure 2: Model Comparison — Marketing and Product Performance Dataset',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/RQ2_Figure2_Model_Comparison.pdf', bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.
